In [1]:
# ============================================================
# 04 - ECONOMIC ANALYSIS
# HOUSEHOLD PURCHASING POWER IN MOROCCO
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

TABLES_DIR = Path("../outputs/tables")
TABLES_DIR.mkdir(parents=True, exist_ok=True)

file_path = (
    "../data/processed/"
    "study_02_household_purchasing_power_enriched_2007_2024.xlsx"
)

df = pd.read_excel(file_path)

print("Shape:", df.shape)

display(df.head())

Shape: (18, 14)


,year,cpi_base_2017_harmonized,disposable_income_s14_harmonized,consumption_s14_harmonized,real_disposable_income,real_consumption,inflation,real_disposable_income_growth,real_consumption_growth,saving_rate,nominal_income_index_2007,nominal_consumption_index_2007,real_income_index_2007,real_consumption_index_2007
0,2007,86.67,481012,414786,"555,022.60","478,606.78",NaN,NaN,NaN,13.77,100.00,100.00,100.00,100.00
1,2008,89.88,529868,462466,"589,540.07","514,547.47",3.71,6.22,7.51,12.72,110.16,111.50,106.22,107.51
2,2009,90.81,567820,485171,"625,295.26","534,280.46",1.03,6.06,3.84,14.56,118.05,116.97,112.66,111.63
3,2010,91.65,589668,509474,"643,364.49","555,867.84",0.93,2.89,4.04,13.60,122.59,122.83,115.92,116.14
4,2011,92.50,621887,539099,"672,315.36","582,814.14",0.92,4.50,4.85,13.31,129.29,129.97,121.13,121.77


In [2]:
# ============================================================
# LONG-TERM EVOLUTION
# ============================================================

first_year = df.iloc[0]
last_year = df.iloc[-1]

summary_2007_2024 = pd.DataFrame({
    "Indicator": [
        "CPI",
        "Nominal disposable income",
        "Nominal consumption",
        "Real disposable income",
        "Real consumption"
    ],
    "2007": [
        first_year["cpi_base_2017_harmonized"],
        first_year["disposable_income_s14_harmonized"],
        first_year["consumption_s14_harmonized"],
        first_year["real_disposable_income"],
        first_year["real_consumption"]
    ],
    "2024": [
        last_year["cpi_base_2017_harmonized"],
        last_year["disposable_income_s14_harmonized"],
        last_year["consumption_s14_harmonized"],
        last_year["real_disposable_income"],
        last_year["real_consumption"]
    ]
})

summary_2007_2024["Change_%"] = (
    (
        summary_2007_2024["2024"]
        / summary_2007_2024["2007"]
        - 1
    )
    * 100
).round(2)

display(summary_2007_2024)

,Indicator,2007,2024,Change_%
0,CPI,86.67,118.70,36.96
1,Nominal disposable income,"481,012.00","1,077,016.00",123.91
2,Nominal consumption,"414,786.00","944,065.00",127.60
3,Real disposable income,"555,022.60","907,342.88",63.48
4,Real consumption,"478,606.78","795,336.98",66.18


In [3]:
summary_2007_2024.to_excel(
    TABLES_DIR / "table_01_long_term_evolution.xlsx",
    index=False
)

print("Table 01 exported successfully.")

Table 01 exported successfully.


In [4]:
# ============================================================
# PERIOD ANALYSIS
# ============================================================

periods = {
    "2007-2014": (2007, 2014),
    "2015-2019": (2015, 2019),
    "2020-2024": (2020, 2024)
}

period_results = []

for period_name, (start, end) in periods.items():

    period_df = df[
        (df["year"] >= start) &
        (df["year"] <= end)
    ].copy()

    first = period_df.iloc[0]
    last = period_df.iloc[-1]

    nominal_income_change = (
        (
            last["disposable_income_s14_harmonized"]
            / first["disposable_income_s14_harmonized"]
            - 1
        )
        * 100
    )

    real_income_change = (
        (
            last["real_disposable_income"]
            / first["real_disposable_income"]
            - 1
        )
        * 100
    )

    consumption_change = (
        (
            last["real_consumption"]
            / first["real_consumption"]
            - 1
        )
        * 100
    )

    average_inflation = period_df["inflation"].mean()

    period_results.append({
        "Period": period_name,
        "Nominal_income_change_%": nominal_income_change,
        "Real_income_change_%": real_income_change,
        "Real_consumption_change_%": consumption_change,
        "Average_inflation_%": average_inflation,
        "Average_saving_rate_%": period_df["saving_rate"].mean()
    })

period_analysis = pd.DataFrame(period_results).round(2)

display(period_analysis)

,Period,Nominal_income_change_%,Real_income_change_%,Real_consumption_change_%,Average_inflation_%,Average_saving_rate_%
0,2007-2014,44.80,30.88,33.52,1.46,13.25
1,2015-2019,12.62,7.76,10.01,1.21,12.59
2,2020-2024,35.78,17.25,21.20,3.16,12.45


In [5]:
period_analysis.to_excel(
    TABLES_DIR / "table_02_period_analysis.xlsx",
    index=False
)

print("Table 02 exported successfully.")

Table 02 exported successfully.


In [6]:
# ============================================================
# IDENTIFY KEY ECONOMIC YEARS
# ============================================================

inflation_data = df.dropna(
    subset=["inflation"]
)

real_income_growth_data = df.dropna(
    subset=["real_disposable_income_growth"]
)

saving_rate_data = df.dropna(
    subset=["saving_rate"]
)

key_years = {
    "Highest inflation year": (
        int(
            inflation_data.loc[
                inflation_data["inflation"].idxmax(),
                "year"
            ]
        ),
        float(inflation_data["inflation"].max())
    ),

    "Lowest real income growth year": (
        int(
            real_income_growth_data.loc[
                real_income_growth_data[
                    "real_disposable_income_growth"
                ].idxmin(),
                "year"
            ]
        ),
        float(
            real_income_growth_data[
                "real_disposable_income_growth"
            ].min()
        )
    ),

    "Highest real income growth year": (
        int(
            real_income_growth_data.loc[
                real_income_growth_data[
                    "real_disposable_income_growth"
                ].idxmax(),
                "year"
            ]
        ),
        float(
            real_income_growth_data[
                "real_disposable_income_growth"
            ].max()
        )
    ),

    "Highest saving rate year": (
        int(
            saving_rate_data.loc[
                saving_rate_data["saving_rate"].idxmax(),
                "year"
            ]
        ),
        float(saving_rate_data["saving_rate"].max())
    ),

    "Lowest saving rate year": (
        int(
            saving_rate_data.loc[
                saving_rate_data["saving_rate"].idxmin(),
                "year"
            ]
        ),
        float(saving_rate_data["saving_rate"].min())
    )
}

for indicator, result in key_years.items():
    print(
        f"{indicator}: "
        f"{result[0]} | "
        f"{result[1]:.2f}"
    )

Highest inflation year: 2022 | 6.64
Lowest real income growth year: 2020 | -4.35
Highest real income growth year: 2021 | 8.09
Highest saving rate year: 2020 | 15.20
Lowest saving rate year: 2023 | 10.54


In [7]:
key_years_table = pd.DataFrame(
    [
        {
            "Indicator": key,
            "Year": value[0],
            "Value": value[1]
        }
        for key, value in key_years.items()
    ]
)

display(key_years_table)

key_years_table.to_excel(
    TABLES_DIR / "table_03_key_years.xlsx",
    index=False
)

,Indicator,Year,Value
0,Highest inflation year,2022,6.64
1,Lowest real income growth year,2020,-4.35
2,Highest real income growth year,2021,8.09
3,Highest saving rate year,2020,15.20
4,Lowest saving rate year,2023,10.54


In [8]:
# ============================================================
# NOMINAL VS REAL GAP
# ============================================================

df["income_inflation_gap"] = (
    df["nominal_income_index_2007"]
    - df["real_income_index_2007"]
)

df["consumption_inflation_gap"] = (
    df["nominal_consumption_index_2007"]
    - df["real_consumption_index_2007"]
)

gap_analysis = df[
    [
        "year",
        "nominal_income_index_2007",
        "real_income_index_2007",
        "income_inflation_gap",
        "nominal_consumption_index_2007",
        "real_consumption_index_2007",
        "consumption_inflation_gap"
    ]
].round(2)

display(gap_analysis)

,year,nominal_income_index_2007,real_income_index_2007,income_inflation_gap,nominal_consumption_index_2007,real_consumption_index_2007,consumption_inflation_gap
0,2007,100.00,100.00,0.00,100.00,100.00,0.00
1,2008,110.16,106.22,3.94,111.50,107.51,3.99
2,2009,118.05,112.66,5.39,116.97,111.63,5.34
3,2010,122.59,115.92,6.67,122.83,116.14,6.69
4,2011,129.29,121.13,8.15,129.97,121.77,8.20
5,2012,134.31,124.24,10.06,136.28,126.07,10.21
6,2013,142.79,129.64,13.15,143.19,130.00,13.19
7,2014,144.80,130.88,13.92,147.72,133.52,14.20
8,2015,152.03,135.27,16.76,151.64,134.93,16.72
9,2016,151.99,133.04,18.95,154.82,135.52,19.30


In [9]:
gap_analysis.to_excel(
    TABLES_DIR / "table_04_nominal_real_gap.xlsx",
    index=False
)

print("Table 04 exported successfully.")

Table 04 exported successfully.
